# 📘 Tutorial 3: Gaussian Processes as Surrogate Models

> This notebook is provided in a clean, non-executed state for readability and reproducibility.

In **Tutorial 2**, we studied why a useful surrogate must represent not only a prediction, but also a measure of uncertainty.

We saw that when the objective is only partially observed:
- a mean curve alone is not enough,
- uncertainty should depend on data support,
- and good optimisation decisions require balancing predicted value with what the model still does not know.

In this tutorial, we take the next conceptual step:

> **where do predictive mean and predictive uncertainty come from in a principled surrogate model?**

The answer is:

> **Gaussian Processes.**

This is a major shift.

In the previous tutorial, uncertainty was introduced heuristically in order to build intuition.
Here, we move to a fully probabilistic framework in which both the predictive mean and predictive uncertainty arise naturally from the model itself.

A Gaussian Process is not just a smoother fitting tool.
It is a **distribution over functions**.

That means it does not begin by committing to one fixed curve.
Instead, it begins with a prior belief over what kinds of functions are plausible, and then updates that belief after observations are incorporated.

Once this is understood, surrogate modelling becomes more than approximating an unknown function.

It becomes a problem of:
- specifying structured assumptions through a kernel,
- turning those assumptions into a prior over functions,
- conditioning that prior on observed data,
- and interpreting the resulting posterior mean and posterior uncertainty as a principled surrogate model.

To make this concrete, we continue with simple one-dimensional black-box objectives and study Gaussian Processes through their:
- kernel structure,
- prior samples,
- posterior mean and uncertainty,
- and sequential updates as new observations are added.

---

**This tutorial is designed to shift perspective**
- from *“uncertainty can be added heuristically to a surrogate”*
- to *“a Gaussian Process provides prediction and uncertainty as part of a coherent probabilistic model.”*

---

**The emphasis is on developing intuition for**
- how a kernel encodes similarity between inputs,
- what it means for a Gaussian Process to be a distribution over functions,
- how a GP prior differs from a GP posterior,
- how observations change both the predictive mean and predictive uncertainty,
- and why Gaussian Processes are such natural surrogate models for Bayesian Optimisation.

---

**Key ideas explored include**
- the RBF kernel as a covariance function,
- Gaussian Process priors and posterior updates,
- prior samples vs posterior samples,
- posterior mean and posterior uncertainty,
- the effect of kernel hyperparameters such as lengthscale and variance,
- and how sequentially adding observations changes the surrogate over time.

---

This tutorial serves as the conceptual bridge from **uncertainty-aware surrogate modelling** to **Bayesian Optimisation with Gaussian Processes**.

In particular, it shows that once the surrogate is probabilistic:
- predictive mean and uncertainty have a principled mathematical origin,
- observations update the whole distribution over functions rather than just one fitted curve,
- and the surrogate becomes a dynamic model that can guide sequential data collection.

These ideas prepare the ground for the next tutorial, where we ask the next natural question:

> **given a Gaussian Process surrogate, where should we evaluate next?**

That is where Bayesian Optimisation begins in full.

---

**Recommended prerequisites**
- Completion of **Tutorials 1–2** in Part 3
- Basic familiarity with surrogate modelling and uncertainty
- Comfort with simple covariance matrices, Gaussian distributions, and plots

---

**Author**: Angze Li

**Last updated**: 2026-04-02

**Version**: v1.0

## 🔧 Setup

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

torch.set_default_dtype(torch.float64)

def set_seed(seed=0):
    torch.manual_seed(seed)
    np.random.seed(seed)

def style_ax(ax):
    for spine in ax.spines.values():
        spine.set_linewidth(1.8)
    ax.tick_params(axis="both", labelsize=14)
    for t in ax.get_xticklabels() + ax.get_yticklabels():
        t.set_fontweight("bold")

set_seed(0)

In [ ]:
def expensive_objective(x):
    return 0.35 * torch.sin(2.2 * x) + 0.15 * torch.cos(5.0 * x) + 0.03 * (x - 1.5) ** 2 - 0.25

x_dense = torch.linspace(-3.0, 3.0, 400)
y_dense = expensive_objective(x_dense)

## 1. Conceptual setup: we still only observe a few evaluations

Before introducing Gaussian Processes, it is useful to restate the basic modelling problem.

Even though this tutorial is more mathematically structured than the previous ones, the setting has not changed:

> the true objective exists, but we do not get to observe it everywhere.

Instead, we only see a small number of evaluations of that objective at selected input locations.

That is the central reason surrogate modelling is needed in the first place.

---

### What the code does

This cell creates a small training dataset by evaluating the objective at 8 evenly spaced points in the interval

$$
[-1.5,\;1.5].
$$

It then plots:
- the full objective curve, for visual reference,
- and the observed data points that will be used to build the surrogate model.

So the figure contrasts:
- the unknown function we would like to understand,
- and the sparse observations that are actually available to us.

---

### Why this matters before introducing Gaussian Processes

A Gaussian Process is still a **surrogate model**.

It is not a way of revealing the whole function directly.
It is a way of reasoning about the function from limited observations.

So this cell plays an important conceptual role:

> it reminds us that the GP is being introduced to solve the same problem as before — learning from sparse, expensive observations.

Everything that follows in the notebook should be interpreted in that light.

The GP prior, the kernel, the posterior mean, and the uncertainty band are all responses to the same underlying situation:

- few observations,
- incomplete information,
- and the need to model what has not yet been observed.

---

### Key takeaway

This figure is the starting point for the notebook:

> we only observe a few evaluations of the objective, and the purpose of a Gaussian Process is to turn those sparse observations into a principled predictive model with uncertainty.

In [ ]:
x_train = torch.linspace(-1.5, 1.5, 8)
y_train = expensive_objective(x_train)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(x_dense.numpy(), y_dense.numpy(), linewidth=2.0, label="True objective")
ax.scatter(x_train.numpy(), y_train.numpy(), s=40, zorder=3, label="Observations")
ax.set_title("We only observe a few evaluations of the objective", fontsize=18, fontweight="bold")
ax.set_xlabel("x", fontsize=22, fontweight="bold")
ax.set_ylabel("f(x)", fontsize=22, fontweight="bold")
ax.legend(prop={"size": 10, "weight": "bold"})
style_ax(ax)
plt.show()

## 2. The RBF kernel: how a Gaussian Process encodes similarity

To build a Gaussian Process, we need a way of specifying how function values at different input locations should be related.

This is the role of the **kernel**.

A kernel is a function

$$
k(x, x')
$$

that measures how similar two input points $x$ and $x'$ are assumed to be.

In a Gaussian Process, this similarity function determines the covariance between the random function values at those two points:

$$
\mathrm{Cov}\bigl(f(x), f(x')\bigr) = k(x, x').
$$

So the kernel is what gives structure to the GP.
It tells the model which points should behave similarly, and which points should be allowed to behave more independently.

---

### What is the mathematical role of the kernel?

A Gaussian Process is a distribution over functions.
To define such a distribution, we need to specify:

- a mean function
  $$
  m(x),
  $$
- and a covariance function
  $$
  k(x, x').
  $$

Together, these define the GP:

$$
f(x) \sim \mathcal{GP}\bigl(m(x), k(x, x')\bigr).
$$

In this tutorial, the kernel is the main object of interest, because it determines the shape, smoothness, and correlation structure of the functions that the GP considers plausible.

If two points $x$ and $x'$ have a large kernel value, then the model expects $f(x)$ and $f(x')$ to be strongly correlated.
If the kernel value is small, then the model allows those function values to vary more independently.

So the kernel is not just a technical detail:

> it is the mathematical object that encodes our assumptions about what kinds of functions are likely.

---

### What is the RBF kernel?

In this notebook, we use the **RBF kernel**, where RBF stands for **Radial Basis Function**.

It is also commonly called the **squared exponential kernel** or the **Gaussian kernel**.

Its form is

$$
k(x, x') = \sigma_f^2 \exp\!\left(-\frac{(x-x')^2}{2\ell^2}\right),
$$

where:

- $\sigma_f^2$ is the **variance** parameter
- $\ell$ is the **lengthscale**
- $(x-x')^2$ is the squared distance between the two input points

This kernel says:

- points that are close together in input space should have highly correlated function values,
- points that are far apart should have weaker correlation.

Because the kernel depends only on the distance between $x$ and $x'$, it is called a **radial** basis function.

---

### What do the parameters mean?

The RBF kernel has two especially important parameters.

#### 1. Variance

The parameter `variance` controls the overall vertical scale of function variation.

Larger variance means:
- larger covariance values overall,
- and therefore function values are allowed to vary more strongly.

In the formula, this appears as

$$
\sigma_f^2.
$$

#### 2. Lengthscale

The parameter `lengthscale` controls how quickly similarity decays with distance.

A small lengthscale means:
- similarity drops quickly as points move apart,
- so functions can vary more rapidly over short distances.

A large lengthscale means:
- similarity decays more slowly,
- so functions are smoother and vary more gradually.

So the lengthscale controls the notion of **how local** the influence of one point should be.

---

### What the code does

The function `rbf_kernel(x1, x2, lengthscale=0.8, variance=1.0)` computes the RBF kernel matrix between two sets of input points.

Here:

- `x1` is a collection of input locations
  $$
  x_1^{(1)}, x_1^{(2)}, \dots, x_1^{(n)}
  $$
- `x2` is another collection of input locations
  $$
  x_2^{(1)}, x_2^{(2)}, \dots, x_2^{(m)}
  $$

So the goal of the function is to compare **every point in `x1` with every point in `x2`**.

The code first reshapes the two inputs into column vectors so that pairwise distances can be computed by broadcasting.

It then forms the squared distance matrix

$$
\left(x_1^{(i)} - x_2^{(j)}\right)^2
$$

for all pairs of points $i$ and $j$.

Next, it applies the RBF formula

$$
k\!\left(x_1^{(i)}, x_2^{(j)}\right)
=
\text{variance}\cdot
\exp\!\left(
-\frac{\left(x_1^{(i)}-x_2^{(j)}\right)^2}{2\,\text{lengthscale}^2}
\right).
$$

So the output is a matrix of shape

$$
n \times m,
$$

whose entry in row $i$ and column $j$ is the kernel value between the $i$-th point in `x1` and the $j$-th point in `x2`.

In other words, the output matrix tells us how similar every point in `x1` is to every point in `x2`.

When `x1 = x2`, this becomes a square covariance matrix for a single set of inputs, which is exactly the structure used by a Gaussian Process.

---

### Why this matters

This cell introduces one of the most important objects in the whole notebook.

The kernel is what turns a Gaussian Process from an abstract distribution into a concrete model of smooth, correlated functions.

Everything that follows — prior samples, posterior mean, posterior uncertainty, and sequential updating — depends on this kernel.

So the key idea to keep in mind is:

> the kernel tells the Gaussian Process what “similar inputs should give similar outputs” means.

That is the mathematical foundation of the surrogate model.

---

### Key takeaway

The RBF kernel is a smooth similarity function of the form

$$
k(x, x') = \sigma_f^2 \exp\!\left(-\frac{(x-x')^2}{2\ell^2}\right),
$$

and in a Gaussian Process it defines the covariance between function values at different inputs.

This means:
- nearby points are assumed to have similar function values,
- far-apart points are less strongly related,
- and the kernel therefore encodes the smoothness assumptions of the surrogate model.

In [ ]:
def rbf_kernel(x1, x2, lengthscale=0.8, variance=1.0):
    x1 = x1.reshape(-1, 1)
    x2 = x2.reshape(-1, 1)
    sqdist = (x1 - x2.T) ** 2
    return variance * torch.exp(-0.5 * sqdist / lengthscale**2)

## 3. Visualising the kernel: matrix view and similarity view

The previous cell introduced the RBF kernel mathematically.

This cell now makes that kernel more concrete by showing it in two complementary ways:

- as a **kernel matrix**
- and as a **similarity function to a fixed point**

Together, these two views help build intuition for what the kernel is actually doing inside a Gaussian Process.

---

### What the code does

The code first creates a small grid of input points and computes the matrix

$$
K_{ij} = k(x_i, x_j),
$$

using the RBF kernel.

This kernel matrix is shown as an image in the left panel.

The same code then chooses three anchor points,

$$
x' \in \{-2.0,\;0.0,\;2.0\},
$$

and for each one plots the function

$$
k(x, x')
$$

over the full input range.

These curves are shown in the right panel.

So the two plots are showing the same kernel from two different but equivalent perspectives.

---

### Left panel: the kernel matrix

The matrix

$$
K_{ij} = k(x_i, x_j)
$$

records the similarity between every pair of input points on the grid.

The bright diagonal appears because every point is maximally similar to itself:

$$
k(x_i, x_i) = \sigma_f^2.
$$

As we move away from the diagonal, the points become farther apart, so their similarity decreases.

This is why the matrix becomes darker away from the diagonal.

So the kernel matrix gives a global picture of the covariance structure:

> nearby inputs are strongly correlated, while distant inputs are less strongly related.

This matrix is exactly the covariance matrix that the Gaussian Process uses when defining distributions over function values.

---

### Right panel: similarity to a fixed point

The curves in the right panel show the kernel as a function of $x$ for several fixed anchor points $x'$.

Each curve is

$$
k(x, x'),
$$

which tells us how similar any input location $x$ is to the chosen anchor point.

These curves are largest at

$$
x = x',
$$

and decay smoothly as $x$ moves away from the anchor.

This makes the meaning of the kernel especially intuitive:

> the kernel says how strongly one input location should influence another.

So the right panel is a local view of the same idea that the matrix shows globally.

---

### Why these two views are useful together

The two panels are showing the same mathematical object, but from different perspectives:

- the **matrix view** shows all pairwise similarities at once,
- the **function view** shows how one point influences the surrounding region.

Together, they help explain why the kernel is so central in a Gaussian Process.

The kernel is not just a numerical trick.
It is the mechanism that tells the model:

- which points should behave similarly,
- how far correlations should extend,
- and how smooth the resulting functions should be.

---

### Key takeaway

This figure makes the role of the kernel visually clear:

- the kernel matrix shows the full covariance structure across the input grid,
- and the curves $k(x, x')$ show how similarity decays away from a given point.

So the kernel is best understood as a structured notion of similarity, and that similarity is what allows a Gaussian Process to model smooth, correlated functions.

In [ ]:
x_grid_small = torch.linspace(-3.0, 3.0, 60)
K = rbf_kernel(x_grid_small, x_grid_small, lengthscale=0.8, variance=1.0)

anchors = torch.tensor([-2.0, 0.0, 2.0])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

im = axes[0].imshow(K.numpy(), origin="lower", aspect="auto")
axes[0].set_title("RBF kernel matrix", fontsize=18, fontweight="bold")
axes[0].set_xlabel("Input index", fontsize=22, fontweight="bold")
axes[0].set_ylabel("Input index", fontsize=22, fontweight="bold")
style_ax(axes[0])
plt.colorbar(im, ax=axes[0])

for x0 in anchors:
    kx = rbf_kernel(x_dense, x0.unsqueeze(0), lengthscale=0.8, variance=1.0).squeeze()
    axes[1].plot(x_dense.numpy(), kx.numpy(), linewidth=2.0, label=f"x'={float(x0):.1f}")

axes[1].set_title("The kernel measures similarity to a point", fontsize=18, fontweight="bold")
axes[1].set_xlabel("x", fontsize=22, fontweight="bold")
axes[1].set_ylabel("k(x, x')", fontsize=22, fontweight="bold")
axes[1].legend(prop={"size": 10, "weight": "bold"}, loc="lower left")
style_ax(axes[1])

plt.tight_layout()
plt.show()

## 4. Sampling from a Gaussian Process prior

Once the kernel has been defined, we can use it to do something much more revealing:

> **sample whole functions from the Gaussian Process prior.**

Here, the GP prior means the model’s distribution over possible functions before any observations have been incorporated. It does not represent a fitted function; rather, it represents the family of functions that the model considers plausible based only on the prior mean and kernel.

This is one of the most important conceptual steps in the notebook.

Up to now, the kernel has been described as a similarity function and as a covariance matrix.
This cell shows what that covariance structure actually means in practice.

A Gaussian Process is not a single function.
It is a **distribution over possible functions**.

So before seeing any observations, the GP prior represents a family of functions that the model considers plausible.

---

### What the code does

The function `sample_gp_prior(...)` generates random samples from a Gaussian Process prior evaluated on a finite input grid.

The steps are:

1. Compute the kernel matrix
   $$
   K_{ij} = k(x_i, x_j),
   $$
   using the RBF kernel.

2. Add a very small diagonal term called `jitter`:
   $$
   K \leftarrow K + \epsilon I.
   $$
   This is mainly a numerical stabilisation step that helps ensure the covariance matrix is well behaved for Cholesky decomposition.

3. Compute a Cholesky factorisation
   $$
   K = LL^\top.
   $$

4. Generate independent standard Gaussian random vectors
   $$
   z \sim \mathcal{N}(0, I).
   $$

5. Transform them into correlated Gaussian samples using
   $$
   Lz.
   $$

The result is a collection of random function values whose covariance structure is exactly determined by the kernel matrix.

---

### The mathematical background

At a finite collection of input points
$$
x_1, x_2, \dots, x_n,
$$
a Gaussian Process prior says that the vector of function values

$$
\begin{pmatrix}
f(x_1) \\
f(x_2) \\
\vdots \\
f(x_n)
\end{pmatrix}
$$

follows a multivariate normal distribution:

$$
f(x_{1:n}) \sim \mathcal{N}(0, K),
$$

where the covariance matrix is

$$
K_{ij} = k(x_i, x_j).
$$

So this function is literally sampling from that multivariate Gaussian distribution.

The Cholesky factorisation works because if
$$
z \sim \mathcal{N}(0, I),
$$
then
$$
Lz \sim \mathcal{N}(0, LL^\top) = \mathcal{N}(0, K).
$$

That is the mathematical reason this code produces valid GP prior samples.

---

### Why this matters

This cell is where the phrase

> **a Gaussian Process is a distribution over functions**

stops being abstract and becomes concrete.

Each sampled curve is one possible function that the GP prior considers plausible.

The shape of those functions is not arbitrary.
It is controlled by the kernel.

Because we are using the RBF kernel, the sampled functions are smooth and correlated across nearby input points.

So this function provides the computational mechanism for turning kernel-based covariance assumptions into actual random functions that can be plotted and interpreted.

---

### Key takeaway

`sample_gp_prior(...)` uses the kernel matrix to generate random function samples from the GP prior.

Mathematically, it is sampling from

$$
\mathcal{N}(0, K),
$$

where
$$
K_{ij} = k(x_i, x_j).
$$

This is the key step that makes Gaussian Processes interpretable as **probabilistic surrogate models over functions**, rather than just deterministic curve-fitting tools.

In [ ]:
def sample_gp_prior(x, lengthscale=0.8, variance=1.0, n_samples=5, jitter=1e-6):
    K = rbf_kernel(x, x, lengthscale=lengthscale, variance=variance)
    K = K + jitter * torch.eye(len(x), dtype=x.dtype)
    L = torch.linalg.cholesky(K)
    z = torch.randn(len(x), n_samples, dtype=x.dtype)
    return (L @ z).T

## 5. Samples from a Gaussian Process prior

Now that the kernel has been defined and we know how to sample from the corresponding multivariate Gaussian distribution, we can finally visualise the **Gaussian Process prior** itself.


A Gaussian Process prior is not a fitted curve, and it is not an estimate of the true objective.

Instead, it is:

> **the model’s belief about what kinds of functions could plausibly exist before any observations have been used to constrain it.**

So when we speak about the **GP prior**, we mean the model’s structured uncertainty about the unknown function *before seeing data*.

---

### What is the GP prior in a real sense?

Suppose we are told only that there is some unknown function $f(x)$, but we have not yet observed any values of it.

At that point, we still want the model to have some sensible assumptions.

For example, we may believe that:

- the function should be reasonably smooth,
- nearby input points should produce similar output values,
- and large erratic jumps over tiny input changes are unlikely.

Those assumptions do not come from data.
They come from the **prior**.

So the GP prior is not a guess of one exact function.
It is a probability distribution over many possible functions, weighted according to how plausible they are under the chosen kernel.

In this notebook, because we are using an RBF kernel, the GP prior is effectively saying:

> before seeing any observations, I believe the unknown function is likely to be smooth and locally correlated.

That is the “real” meaning of the prior.

---

### What the code does

The code draws 5 random sample functions from the GP prior over the dense input grid `x_dense`.

Each sample is one possible realisation of the unknown function under the prior assumptions encoded by the RBF kernel.

So the plotted curves are **not fitted to data**.
They are simply random functions that the model considers plausible *before conditioning on any observations*.

The legend uses a single label, `GP prior samples`, because the individual curves are not meant to be interpreted separately.
They are all examples drawn from the same prior distribution.

---

### The mathematical meaning of the plot

At the finite collection of grid points

$$
x_1, x_2, \dots, x_n,
$$

the GP prior says that the vector of function values

$$
\begin{pmatrix}
f(x_1) \\
f(x_2) \\
\vdots \\
f(x_n)
\end{pmatrix}
$$

follows a multivariate normal distribution:

$$
f(x_{1:n}) \sim \mathcal{N}(0, K),
$$

where the covariance matrix is given by

$$
K_{ij} = k(x_i, x_j).
$$

So each plotted curve is one draw from that joint Gaussian distribution over function values on the grid.

Because the kernel is smooth, the sampled functions are smooth.
Because nearby inputs are strongly correlated, nearby parts of the curve tend to move together.

So the visual appearance of the prior samples is a direct consequence of the kernel structure.

---

### Why this matters conceptually

This figure makes the phrase

> **a Gaussian Process is a distribution over functions**

fully concrete.

Before this cell, that phrase is easy to repeat but still somewhat abstract.

After this cell, it becomes visual and real:

- one sampled curve = one possible function
- several sampled curves = different functions the model considers plausible
- the whole collection = the prior belief over the unknown objective

This is fundamentally different from ordinary deterministic curve fitting.

A Gaussian Process does not begin by committing to one function.
It begins by expressing uncertainty over many functions.

That is why it is such a natural surrogate model for Bayesian Optimisation.

---

### Why the prior is important before seeing data

A common question is: why do we even need a prior if the goal is to learn from observations later?

The answer is that sparse observations alone are never enough to determine a unique function.

Even after seeing a few data points, there are infinitely many functions that could still be consistent with them.

So the model must have some prior rule for deciding:

- what kinds of functions are plausible,
- how information should spread from one observed point to nearby points,
- and how uncertain it should remain in regions with little or no data.

The GP prior provides exactly that starting structure.

So the prior matters because it tells the model how to behave *before data* and therefore how to generalise *once data arrives*.

---

### What this figure is and is not showing

This figure is showing:

- the model’s pre-data belief over possible functions,
- shaped by the RBF kernel.

It is **not** showing:

- a fit to the true objective,
- a posterior after observing data,
- or the final surrogate we will use for optimisation.

So the correct interpretation is:

> these are plausible functions according to the prior assumptions alone.

In the next section, we will see how this prior gets updated once actual observations are provided.

That update is what produces the **GP posterior**, with a mean prediction and uncertainty band that are now informed by data.

---

### Key takeaway

> A GP prior is the model’s probability distribution over possible functions before seeing any data.

The curves in this figure are random samples from that prior.

They are important because they show, in a concrete way, that a Gaussian Process is not just a fitting tool — it is a probabilistic model of what the unknown function could be like before observations are used to constrain it.

In [ ]:
prior_samples = sample_gp_prior(x_dense, lengthscale=0.8, variance=1.0, n_samples=5)

fig, ax = plt.subplots(figsize=(8, 5))
for i, s in enumerate(prior_samples):
    label = "GP prior samples" if i == 0 else None
    ax.plot(x_dense.numpy(), s.numpy(), linewidth=2.0, alpha=0.9, label=label)

ax.set_title("Samples from a Gaussian Process prior", fontsize=18, fontweight="bold")
ax.set_xlabel("x", fontsize=22, fontweight="bold")
ax.set_ylabel("f(x)", fontsize=22, fontweight="bold")
ax.legend(prop={"size": 10, "weight": "bold"})
style_ax(ax)
plt.show()

## 6. Kernel hyperparameters shape the Gaussian Process prior

The previous prior-sample figure showed that a Gaussian Process does not represent a single function, but a distribution over many possible functions.

A natural next question is:

> **How do the kernel hyperparameters change what kinds of functions the GP prior considers plausible?**

This cell answers that question by varying the two most important hyperparameters of the RBF kernel:

- the **lengthscale**
- the **variance**

and comparing the resulting prior samples side by side.

---

### What the code does

We consider four combinations of kernel hyperparameters:

$$
(\ell, \sigma_f^2) \in
\{(0.3, 0.5),\; (0.3, 1.5),\; (1.2, 0.5),\; (1.2, 1.5)\}.
$$

For each pair:

- a Gaussian Process prior is defined using the RBF kernel,
- 5 sample functions are drawn from that prior,
- and the resulting curves are plotted in one panel of a $2\times2$ figure.

So all four panels show prior samples, but under different assumptions about smoothness and vertical scale.

---

### What the lengthscale controls

Recall that the RBF kernel is

$$
k(x, x') = \sigma_f^2 \exp\left(-\frac{(x-x')^2}{2\ell^2}\right).
$$

The lengthscale parameter $\ell$ controls how quickly correlation decays with distance.

- A **small lengthscale** means that similarity drops quickly as points move apart.
  This allows the sampled functions to change more rapidly over short distances.
- A **large lengthscale** means that similarity decays more slowly.
  This makes the sampled functions smoother and more slowly varying.

So, in practical terms:

> **lengthscale controls how wiggly or smooth the prior sample functions look.**

In the figure:
- the panels with $\ell = 0.3$ should show more rapidly varying behaviour,
- while the panels with $\ell = 1.2$ should show smoother, more slowly varying functions.

---

### What the variance controls

The variance parameter $\sigma_f^2$ controls the overall vertical scale of the function values.

- A **smaller variance** means the sampled functions tend to stay closer to zero.
- A **larger variance** means the sampled functions can vary more strongly in amplitude.

So, in practical terms:

> **variance controls the typical size of the function fluctuations.**

In the figure:
- the panels with variance $0.5$ should show smaller-amplitude curves,
- while the panels with variance $1.5$ should show larger vertical excursions.

---

### How to read the four-panel figure

This layout is useful because it separates the effects of the two hyperparameters.

- Comparing **top vs bottom** isolates the effect of **lengthscale**
- Comparing **left vs right** isolates the effect of **variance**

So the figure lets us see that the kernel is doing more than just “making things smooth.”

It is controlling two different aspects of the prior:

- **how quickly functions vary in $x$**
- **how strongly functions vary in magnitude**

That is why kernel hyperparameters are so important in Gaussian Process modelling.

---

### Why this matters conceptually

This figure makes a deep point about Gaussian Processes:

> the GP prior is not fixed once and for all — it depends on the assumptions built into the kernel.

Changing the hyperparameters changes the family of functions the model considers plausible *before seeing data*.

So the prior is not just a decorative starting point.
It is the mathematical expression of our modelling assumptions.

If the lengthscale is too small, the prior may allow functions that are unrealistically rough.
If the lengthscale is too large, the prior may force the model to be overly smooth.
Similarly, changing the variance changes how much function variation the model expects.

So the kernel hyperparameters shape the space of candidate functions even before any observations are used.

---

### Key takeaway

> The kernel hyperparameters determine what kinds of functions the GP prior considers plausible.

In the RBF kernel:

- the **lengthscale** controls smoothness and correlation range,
- the **variance** controls the overall amplitude of variation.

This is why choosing a kernel is not just a technical detail.
It is a modelling decision that directly affects the surrogate’s prior beliefs about the unknown objective.

In [ ]:
settings = [
    (0.3, 0.5),
    (0.3, 1.5),
    (1.2, 0.5),
    (1.2, 1.5),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True, sharey=True)
axes = axes.flatten()

for ax, (ell, var) in zip(axes, settings):
    samples = sample_gp_prior(x_dense, lengthscale=ell, variance=var, n_samples=5)
    for i, s in enumerate(samples):
        label = "GP prior samples" if i == 0 else None
        ax.plot(x_dense.numpy(), s.numpy(), linewidth=2.0, alpha=0.9, label=label)

    ax.set_title(f"lengthscale={ell}, variance={var}", fontsize=18, fontweight="bold")
    ax.set_xlabel("x", fontsize=22, fontweight="bold")
    ax.set_ylabel("f(x)", fontsize=22, fontweight="bold")
    style_ax(ax)

axes[0].legend(prop={"size": 10, "weight": "bold"}, loc="upper left")
plt.tight_layout()
plt.show()

## 7. Computing the Gaussian Process posterior

Up to this point, the notebook has focused on the **GP prior**: the model’s belief about what kinds of functions are plausible before any observations are used.

The next step is the most important one in Gaussian Process regression:

> **update that prior using observed data to obtain a posterior predictive distribution.**

This is the moment where the Gaussian Process becomes a usable surrogate model.

After conditioning on observed data, the GP no longer represents just generic plausible functions.
It now represents functions that are consistent with the observations, while still retaining uncertainty away from them.

---

### What the code does

The function `gp_posterior(...)` computes the predictive mean, predictive variance, and predictive covariance of a Gaussian Process at test inputs `x_test`, conditioned on the training data

$$
\mathcal{D} = \{(x_i, y_i)\}_{i=1}^n.
$$

It takes as input:

- `x_train`: the observed input locations
- `y_train`: the observed function values at those locations
- `x_test`: the input locations where we want to make predictions
- kernel hyperparameters: `lengthscale` and `variance`
- a small `noise` term for numerical stability and regularisation

The function returns:

- `mu`: the GP posterior mean at the test points
- `var`: the diagonal of the posterior covariance matrix
- `cov`: the full posterior covariance matrix

So this function is the core computational step that turns a GP into a predictive model.

---

### The kernel matrices

The first three lines build the key kernel matrices:

$$
K_{xx} = k(x_{\mathrm{train}}, x_{\mathrm{train}}),
$$

$$
K_{xs} = k(x_{\mathrm{train}}, x_{\mathrm{test}}),
$$

$$
K_{ss} = k(x_{\mathrm{test}}, x_{\mathrm{test}}).
$$

These matrices play different roles:

- $K_{xx}$ describes the covariance among the observed training points
- $K_{xs}$ describes the covariance between training points and test points
- $K_{ss}$ describes the covariance among the test points themselves

Together, they determine how information from the observed data is transferred to new input locations.

---

### Why the noise term is added

The line

$$
K_{xx} \leftarrow K_{xx} + \sigma_n^2 I
$$

adds a small diagonal term to the training covariance matrix.

This serves two purposes:

- it improves numerical stability, making the matrix easier to invert or factorise
- it can also be interpreted as a small observation-noise model

In this notebook, the main role is practical:
it ensures the matrix is well behaved for Cholesky decomposition.

---

### The posterior mean

The posterior mean is computed as

$$
\mu(x_*) = K_{xs}^\top (K_{xx} + \sigma_n^2 I)^{-1} y_{\mathrm{train}}.
$$

It can be interpreted as a covariance-weighted combination of the observed function values. Training points that are more similar to the test point contribute more strongly, while the inverse training covariance corrects for redundancy among observations.

As a result, the posterior mean gives the GP’s best estimate of the function after incorporating both the observed data and the smoothness assumptions encoded by the kernel.

In the code, this is implemented using a Cholesky solve rather than forming the inverse explicitly.


So `mu` is the predictive mean:
- it interpolates and generalises from the training observations
- and it becomes the surrogate’s central prediction curve

---

### The posterior covariance

The posterior covariance is computed as

$$
\Sigma(x_*) = K_{ss} - K_{xs}^\top (K_{xx} + \sigma_n^2 I)^{-1} K_{xs}.
$$

In the code, this appears as

$$
\text{cov} = K\_ss - v^\top v
$$

after solving for the intermediate quantity

$$
v = L^{-1} K_{xs}.
$$

This covariance matrix describes how uncertain the GP remains at the test points *after conditioning on the observed data*.

Its diagonal gives the predictive variances:

$$
\mathrm{Var}(f(x_*)) = \operatorname{diag}(\Sigma(x_*)).
$$

That is why the code extracts

$$
\text{var = diag(cov)}.
$$

So:
- `cov` contains the full posterior uncertainty structure
- `var` gives the pointwise uncertainty used for plotting uncertainty bands

---

### Why Cholesky decomposition is used

The code uses the Cholesky factorisation

$$
K_{xx} + \sigma_n^2 I = LL^\top
$$

instead of directly computing a matrix inverse.

This is standard practice because:
- it is numerically more stable
- it is computationally more efficient
- and it avoids unnecessary floating-point errors

So although the mathematical formulas are written using matrix inverses, the implementation uses Cholesky-based linear algebra for reliability.

---

### What this function means conceptually

This function is the bridge from the GP prior to the GP posterior.

Before observing data, the GP represents a broad family of plausible functions.

After observing data, the GP updates that belief by:
- shifting the mean toward functions that match the observations
- reducing uncertainty near observed points
- and preserving more uncertainty in regions with weaker support

So this function is doing the actual Bayesian update that makes Gaussian Processes useful as surrogate models.

---

### Key takeaway

`gp_posterior(...)` computes the GP posterior predictive distribution from observed data.

Mathematically, it produces:

- a posterior mean
  $$
  \mu(x_*)
  $$
- and a posterior covariance
  $$
  \Sigma(x_*)
  $$

conditioned on the training observations.

This is the core of Gaussian Process regression, because it gives exactly the two quantities needed for surrogate modelling:

- a prediction of the unknown function
- and a principled uncertainty estimate around that prediction

In [ ]:
def gp_posterior(x_train, y_train, x_test, lengthscale=0.8, variance=1.0, noise=1e-4):
    K_xx = rbf_kernel(x_train, x_train, lengthscale=lengthscale, variance=variance)
    K_xs = rbf_kernel(x_train, x_test, lengthscale=lengthscale, variance=variance)
    K_ss = rbf_kernel(x_test, x_test, lengthscale=lengthscale, variance=variance)

    K_xx = K_xx + noise * torch.eye(len(x_train), dtype=x_train.dtype)

    L = torch.linalg.cholesky(K_xx)
    alpha = torch.cholesky_solve(y_train.unsqueeze(1), L)

    mu = (K_xs.T @ alpha).squeeze()

    v = torch.linalg.solve_triangular(L, K_xs, upper=False)
    cov = K_ss - v.T @ v
    var = torch.clamp(torch.diag(cov), min=1e-12)

    return mu, var, cov

## 8. The Gaussian Process posterior: prediction after conditioning on data

The previous sections introduced the key ingredients of a Gaussian Process:

- a kernel that defines similarity,
- a prior distribution over possible functions,
- and a posterior update rule that conditions this prior on observed data.

This cell is the first place where those pieces come together visually.

Here, we plot the **Gaussian Process posterior** itself: the predictive mean together with its uncertainty band after conditioning on the observed evaluations.

This is one of the most important figures in the notebook, because it shows how a GP turns sparse observations into a principled surrogate model.

---

### What the code does

Using the training data

$$
\{(x_i, y_i)\}_{i=1}^n,
$$

the code computes the Gaussian Process posterior at the dense test grid `x_dense` by calling

$$
\texttt{gp\_posterior}(x_{\mathrm{train}}, y_{\mathrm{train}}, x_{\mathrm{test}}).
$$

This returns:

- the posterior mean
  $$
  \mu(x),
  $$
- the posterior variance
  $$
  \mathrm{Var}(f(x)),
  $$
- and the full posterior covariance matrix.

The code then takes

$$
\sigma(x) = \sqrt{\mathrm{Var}(f(x))}
$$

and plots:

- the true objective, for reference,
- the GP posterior mean,
- the uncertainty band
  $$
  \mu(x) \pm 2\sigma(x),
  $$
- and the observed data points.

So this figure shows the GP’s predictive distribution after it has incorporated the available observations.

---

### What is the posterior mean here?

The dashed curve is the GP posterior mean

$$
\mu(x).
$$

This is the model’s best estimate of the unknown objective after conditioning on the observed data.

It is not just an arbitrary smooth fit.
It is the conditional mean of the Gaussian Process posterior, meaning:

> **the expected function value at each input location, given the observations and the kernel assumptions.**

So the posterior mean combines two things:

- information from the observed points,
- and the smoothness structure encoded by the kernel.

That is why the curve is influenced strongly by nearby observations, but still remains smooth between them.

---

### What is the uncertainty band here?

The shaded region is the posterior uncertainty band

$$
\mu(x) \pm 2\sigma(x),
$$

where

$$
\sigma(x) = \sqrt{\mathrm{Var}(f(x))}.
$$

This band represents the GP’s remaining uncertainty after seeing the data.

Its meaning is not that the model has become certain everywhere.
Instead, it is telling us where the observations have constrained the function strongly, and where uncertainty remains.

So the figure should be read in two layers:

- the dashed line tells us what the model currently predicts,
- the band tells us how confident the model is in that prediction.

---

### How to interpret the figure

Several important things are happening at once.

#### 1. The mean is pulled toward the observations
Near the observed data points, the posterior mean is strongly influenced by the known function values.

This is the effect of conditioning:
the GP is no longer sampling arbitrary plausible functions from the prior, but is now restricted to functions that are consistent with the observations.

#### 2. The uncertainty is smaller near observed points
The band narrows in the region where data are available.

This reflects the fact that the model has learned more there.

#### 3. The uncertainty grows away from the observed region
As we move farther from the training data, the band becomes wider.

This is exactly what we want:
the GP does not pretend to know the function equally well everywhere.

So the figure shows the key strength of the GP posterior:

> **it gives both a prediction and a principled measure of how uncertain that prediction still is.**

---

### Why this is better than the heuristic uncertainty in Tutorial 2

In Tutorial 2, uncertainty was introduced heuristically in order to build intuition.

That was useful, but it was not yet a fully probabilistic model.

Here, the uncertainty band is no longer manually designed.
It comes directly from the posterior covariance of the Gaussian Process.

That means the pair

$$
\mu(x), \sigma(x)
$$

now has a principled mathematical origin.

This is the crucial transition:

- in Tutorial 2, uncertainty was an intuitive modelling device
- in Tutorial 3, uncertainty becomes part of a coherent probabilistic surrogate model

---

### Why this matters for optimisation

This figure shows why Gaussian Processes are so useful in Bayesian Optimisation.

A good surrogate model needs to provide:

- a predictive mean, so we know where the objective might be low
- a predictive uncertainty, so we know where the model is still unsure

The GP posterior gives both naturally.

So this figure is not just a regression plot.
It is the first direct demonstration that a GP produces exactly the two objects needed for uncertainty-aware optimisation:

$$
\mu(x)
\qquad\text{and}\qquad
\sigma(x).
$$

---

### Key takeaway

> The Gaussian Process posterior turns sparse observations into a predictive mean and a principled uncertainty band.

The mean
$$
\mu(x)
$$
is the GP’s best estimate of the unknown function after conditioning on the data, while the uncertainty band
$$
\mu(x)\pm 2\sigma(x)
$$
shows where the model is more or less confident.

This is the central reason Gaussian Processes are such powerful surrogate models: they do not only predict what the function might be, but also how certain that prediction is across the domain.

In [ ]:
mu_gp, var_gp, cov_gp = gp_posterior(x_train, y_train, x_dense, lengthscale=1.0, variance=1.0, noise=1e-4)
sigma_gp = torch.sqrt(var_gp)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(x_dense.numpy(), y_dense.numpy(), linewidth=2.0, label="True objective")
ax.plot(x_dense.numpy(), mu_gp.numpy(), linewidth=2.0, linestyle="--", label="GP mean")
ax.fill_between(
    x_dense.numpy(),
    (mu_gp - 2.0 * sigma_gp).numpy(),
    (mu_gp + 2.0 * sigma_gp).numpy(),
    alpha=0.2,
    label=r"$\mu(x)\pm 2\sigma(x)$"
)
ax.scatter(x_train.numpy(), y_train.numpy(), s=40, zorder=3, label="Observations")
ax.set_title("Gaussian Process posterior", fontsize=18, fontweight="bold")
ax.set_xlabel("x", fontsize=22, fontweight="bold")
ax.set_ylabel("f(x)", fontsize=22, fontweight="bold")
ax.legend(prop={"size": 10, "weight": "bold"})
style_ax(ax)
plt.show()

## 9. Posterior uncertainty as a principled quantity

The previous figure showed the Gaussian Process posterior mean together with its uncertainty band.

This cell now isolates the uncertainty term itself by plotting

$$
\sigma(x),
$$

the posterior standard deviation across the input domain.

Looking at $\sigma(x)$ directly makes it easier to see where the model is confident and where substantial uncertainty remains.

---

### What the code does

The code plots the posterior standard deviation

$$
\sigma(x) = \sqrt{\mathrm{Var}(f(x))},
$$

computed from the diagonal of the GP posterior covariance matrix.

So at each input location $x$, the plotted curve tells us how uncertain the Gaussian Process remains about the function value after conditioning on the observed data.

This is not a fitted mean curve.
It is a direct visualisation of the model’s remaining uncertainty.

---

### How to interpret the figure

The main pattern to notice is:

- uncertainty is lower near observed data,
- uncertainty is higher away from observed data.

This is exactly what we would expect from a sensible surrogate model.

Near the training points, the observations strongly constrain the function, so the GP becomes more confident.

Farther away from the observed region, the model has less direct information, so the posterior uncertainty increases.

So the plot gives a clean picture of where the model has learned a lot and where it is still relatively unsure.

---

### Connection to Tutorial 2

This figure should also be read in relation to the uncertainty-band figures from **Tutorial 2**.

There, we introduced a heuristic uncertainty band to build intuition about the idea that confidence should depend on data support.

That was useful conceptually, but it was still a hand-designed construction.

Here, the situation is different.

> Unlike the heuristic uncertainty band in Tutorial 2, the GP uncertainty here is derived directly from the posterior covariance. It is therefore not an arbitrary visual device, but a mathematically grounded measure of remaining uncertainty under the model.

So this plot is the principled version of the earlier idea.

In Tutorial 2, uncertainty was introduced to motivate why prediction alone is incomplete.
In this tutorial, uncertainty now arises naturally from the Gaussian Process itself.

---

### Why this matters mathematically

For the GP posterior, the predictive distribution at the test points is

$$
f(x_*) \mid \mathcal{D} \sim \mathcal{N}(\mu(x_*), \Sigma(x_*)).
$$

The quantity plotted here comes from the diagonal of the posterior covariance matrix:

$$
\sigma(x_*) = \sqrt{\operatorname{diag}(\Sigma(x_*))}.
$$

So $\sigma(x)$ is not chosen heuristically or added by hand.
It is part of the full posterior distribution.

That is why it carries real mathematical meaning:
it describes how much uncertainty remains at each point after the prior has been updated with observed data.

---

### Why this matters for optimisation

This is one of the central reasons Gaussian Processes are so useful in Bayesian Optimisation.

A good surrogate should not only estimate where the objective may be low.
It should also identify where the model is still uncertain.

This uncertainty plot tells us exactly that.

So even before introducing acquisition functions, this figure is already showing one of the essential ingredients of BO:

> the model should know not only what it predicts, but also where it still needs more information.

---

### Key takeaway

> The GP posterior uncertainty is a mathematically grounded measure of remaining uncertainty, not a hand-crafted visual band.

By plotting $\sigma(x)$ directly, we can see where the observations have made the model confident and where substantial uncertainty still remains.

This is the principled counterpart of the heuristic uncertainty ideas introduced in Tutorial 2, and it is one of the key reasons Gaussian Processes are such powerful surrogate models.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(x_dense.numpy(), sigma_gp.numpy(), linewidth=2.5)
ax.set_title("Posterior uncertainty", fontsize=18, fontweight="bold")
ax.set_xlabel("x", fontsize=22, fontweight="bold")
ax.set_ylabel(r"$\sigma(x)$", fontsize=22, fontweight="bold")
style_ax(ax)
plt.show()

## 10. Prior samples vs posterior samples

Earlier in the notebook, we sampled functions from the **GP prior** to visualise what the model considers plausible before seeing any data.

We now do the analogous thing for the **GP posterior**.

This comparison is extremely important, because it shows what conditioning on data actually does to the distribution over functions.

The prior and posterior are both distributions over functions, but they represent fundamentally different states of knowledge:

- the **prior** reflects belief before observing data
- the **posterior** reflects updated belief after incorporating the observations

This cell makes that contrast directly visible.

---

### What the code does

The function
```python
sample_gp_posterior(mu, cov, n_samples=5, jitter=1e-8)
```
generates random samples from a multivariate Gaussian distribution with mean

$$
\mu
$$

and covariance

$$
\Sigma.
$$

In this case:

- `mu` is the GP posterior mean on the test grid
- `cov` is the GP posterior covariance on the same grid

So the sampled curves are draws from the Gaussian Process posterior after conditioning on the observed data.

The code then builds a side-by-side comparison:

- the left panel shows samples from the **GP prior**
- the right panel shows samples from the **GP posterior**

The observed points are also plotted in the posterior panel to show what the posterior is being conditioned on.

---

### How posterior sampling works mathematically

At the test points, the GP posterior is a multivariate normal distribution:

$$
f(x_*) \mid \mathcal{D} \sim \mathcal{N}(\mu(x_*), \Sigma(x_*)).
$$

So the posterior is no longer centred at zero.
It is centred at the posterior mean

$$
\mu(x_*),
$$

and its uncertainty structure is encoded by the posterior covariance

$$
\Sigma(x_*).
$$

The code samples from this distribution by:

1. adding a small jitter term for numerical stability
2. computing a Cholesky factorisation
   $$
   \Sigma = LL^\top
   $$
3. drawing standard Gaussian random vectors
4. transforming them using
   $$
   \mu + Lz
   $$

So, just like in the prior case, each plotted curve is one draw from a multivariate Gaussian distribution — but now it is the **posterior** Gaussian, not the prior.

---

### How to interpret the figure

The left panel shows functions sampled from the prior.

These curves are unconstrained by observations.
They reflect only the smoothness and correlation assumptions encoded by the kernel.

So the prior samples can vary freely across the domain, because the model has not yet been told anything about the true function values.

The right panel shows functions sampled from the posterior.

These curves are no longer arbitrary plausible functions.
They are now constrained by the observed data.

That means:

- they tend to pass near the observed points
- they vary much less in the data-supported region
- and they still retain some uncertainty away from the observations

This is the key point of the figure:

> conditioning on data does not collapse the Gaussian Process to a single function — it turns a broad prior distribution into a more concentrated posterior distribution.

So the posterior is still a distribution over functions, but now it is a much more informed one.

---

### What changes from prior to posterior?

This comparison makes three important changes visible.

#### 1. The centre of the distribution changes
The prior is centred around the prior mean, which in this notebook is effectively zero.

The posterior is centred around the posterior mean

$$
\mu(x),
$$

which is pulled toward the observed data.

#### 2. The spread changes
The prior allows large variation everywhere according to the kernel.

The posterior has reduced spread in regions where observations constrain the function.

So the posterior distribution is narrower where the model has learned more.

#### 3. The sampled functions become data-consistent
Prior samples are just plausible smooth functions.

Posterior samples are plausible smooth functions **that also agree with the observed data**.

That is the real effect of Bayesian conditioning.

---

### Why this matters conceptually

This cell is one of the strongest demonstrations of what a Gaussian Process actually is.

It shows that the GP is not simply producing:
- one fitted curve
- plus one uncertainty band

It is maintaining a full probability distribution over functions, and that distribution changes when data is observed.

So the posterior mean and posterior variance are not isolated outputs.
They are summaries of a richer posterior distribution over possible functions.

That is why Gaussian Processes are such a natural probabilistic surrogate model.

---

### Why this matters for Bayesian Optimisation

Bayesian Optimisation depends on the idea that the model should retain uncertainty even after observing data.

If the model collapsed immediately to one fixed function, there would be no principled way to trade off:
- what currently looks good
- against what is still uncertain

This figure shows why that is possible.

The posterior is not deterministic.
It is still a distribution over plausible functions, only now one that is informed by the observed data.

That retained uncertainty is exactly what later drives acquisition functions.

---

### Key takeaway

> The Gaussian Process posterior is still a distribution over functions, but one that has been updated and constrained by the observed data.

Compared with the prior:

- the posterior samples are centred around the learned mean
- they are more concentrated where data has been observed
- and they remain uncertain in less constrained regions

This is the essence of Bayesian updating in Gaussian Processes: the model does not stop being probabilistic after seeing data — it becomes probabilistic in a more informed way.

In [ ]:
def sample_gp_posterior(mu, cov, n_samples=5, jitter=1e-8):
    cov = cov + jitter * torch.eye(len(mu), dtype=mu.dtype)
    L = torch.linalg.cholesky(cov)
    z = torch.randn(len(mu), n_samples, dtype=mu.dtype)
    return (mu.unsqueeze(1) + L @ z).T

posterior_samples = sample_gp_posterior(mu_gp, cov_gp, n_samples=5)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

prior_samples = sample_gp_prior(x_dense, lengthscale=0.8, variance=1.0, n_samples=5)

for s in prior_samples:
    axes[0].plot(x_dense.numpy(), s.numpy(), linewidth=2.0)
axes[0].set_title("GP prior samples", fontsize=18, fontweight="bold")
axes[0].set_xlabel("x", fontsize=22, fontweight="bold")
axes[0].set_ylabel("f(x)", fontsize=22, fontweight="bold")
style_ax(axes[0])

for s in posterior_samples:
    axes[1].plot(x_dense.numpy(), s.numpy(), linewidth=2.0)
axes[1].scatter(x_train.numpy(), y_train.numpy(), s=40, zorder=3)
axes[1].set_title("GP posterior samples", fontsize=18, fontweight="bold")
axes[1].set_xlabel("x", fontsize=22, fontweight="bold")
axes[1].set_ylabel("f(x)", fontsize=22, fontweight="bold")
style_ax(axes[1])

plt.tight_layout()
plt.show()

## 11. More observations improve the Gaussian Process posterior

The previous sections introduced the Gaussian Process posterior as a predictive mean together with a principled uncertainty band.

A natural next question is:

> **What happens when we increase the number of observations?**

This cell addresses that question by comparing two posterior models built from different data budgets.

---

### What the code does

The code considers two training sets:

- 6 observations
- 16 observations

In both cases, the observations are placed across the interval

$$
[-2,\;2].
$$

For each dataset, the code:

- evaluates the true objective at the chosen observation points,
- computes the GP posterior mean
  $$
  \mu(x),
  $$
- computes the posterior standard deviation
  $$
  \sigma(x),
  $$
- and plots the resulting posterior mean and uncertainty band
  $$
  \mu(x)\pm 2\sigma(x).
  $$

So the figure compares how the Gaussian Process behaves when it has access to a smaller or larger number of observations over the same broad region of the domain.

---

### How to interpret the figure

The main comparison is between the two panels.

With **6 observations**:

- the posterior mean captures the broad shape of the objective,
- but it is still only loosely constrained between observations,
- and the uncertainty band remains relatively wide in several parts of the domain.

With **16 observations**:

- the posterior mean tracks the true objective much more closely,
- the data constrain the function more strongly,
- and the uncertainty band becomes noticeably narrower across much more of the observed region.

So the figure illustrates a very important point:

> **more observations make the posterior both more accurate and more confident.**

---

### What changes mathematically?

The Gaussian Process posterior depends directly on the observed dataset.

When more observations are added:

- the training covariance matrix
  $$
  K_{xx}
  $$
  contains more information,
- the posterior mean
  $$
  \mu(x)
  $$
  is constrained by more known function values,
- and the posterior covariance
  $$
  \Sigma(x)
  $$
  is reduced more strongly across the observed region.

That is why increasing the number of data points changes both:

- the central prediction,
- and the amount of uncertainty remaining around that prediction.

So the effect of additional data is not just to move the mean curve.
It is also to reduce the range of plausible functions that remain consistent with the observations.

---

### Why this is an important GP lesson

This cell makes a key feature of Gaussian Processes very clear:

> **the posterior improves as information accumulates.**

The GP is not fixed after one fit.
It is a model that can be updated repeatedly as more data are collected.

That means every additional evaluation has two benefits:

- it can improve the mean prediction,
- and it can reduce uncertainty in the regions that are now better observed.

This is exactly why Gaussian Processes are so useful for expensive optimisation.

They do not only provide a prediction.
They also show how that prediction sharpens as the data budget increases.

---

### Connection to the next idea

This figure compares two posteriors built from different total numbers of observations.

The next logical step is even more interesting:

> instead of comparing 6 points versus 16 points all at once, what happens when we add **one strategically chosen new observation**?

That is the natural sequential version of the same idea, and it leads directly toward Bayesian Optimisation.

---

### Key takeaway

> Increasing the number of observations improves the Gaussian Process posterior in two coupled ways: the posterior mean becomes more data-informed, and the posterior uncertainty shrinks across the observed region.

This is a central reason Gaussian Processes work so well as surrogate models: they become progressively more useful as new evaluations are added.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for ax, n in zip(axes, [6, 16]):
    x_obs = torch.linspace(-2, 2, n)
    y_obs = expensive_objective(x_obs)
    mu_n, var_n, _ = gp_posterior(x_obs, y_obs, x_dense, lengthscale=0.8, variance=1.0, noise=1e-4)
    sigma_n = torch.sqrt(var_n)

    ax.plot(x_dense.numpy(), y_dense.numpy(), linewidth=2.0, label="true objective")
    ax.plot(x_dense.numpy(), mu_n.numpy(), linewidth=2.0, linestyle="--", label="GP mean")
    ax.fill_between(
        x_dense.numpy(),
        (mu_n - 2.0 * sigma_n).numpy(),
        (mu_n + 2.0 * sigma_n).numpy(),
        alpha=0.2
    )
    ax.scatter(x_obs.numpy(), y_obs.numpy(), s=40, zorder=3)
    ax.set_title(f"{n} observations", fontsize=18, fontweight="bold")
    ax.set_xlabel("x", fontsize=22, fontweight="bold")
    ax.set_ylabel("f(x)", fontsize=22, fontweight="bold")
    style_ax(ax)

plt.tight_layout()
plt.show()

## 12. Kernel hyperparameters shape the Gaussian Process posterior

So far, the notebook has shown that the Gaussian Process posterior is determined by two ingredients:

- the observed data
- the kernel assumptions

This immediately raises an important modelling question:

> **How do the kernel hyperparameters affect the posterior mean and posterior uncertainty after conditioning on data?**

This cell answers that question by varying the two main hyperparameters of the RBF kernel:

- the **lengthscale**
- the **variance**

and comparing the resulting posteriors side by side.

---

### What the code does

We consider four combinations of hyperparameters:

$$
(\ell, \sigma_f^2) \in
\{(0.3, 0.5),\; (0.3, 1.5),\; (1.2, 0.5),\; (1.2, 1.5)\}.
$$

For each pair:

- a GP posterior is computed from the same training data,
- the posterior mean
  $$
  \mu(x)
  $$
  is plotted,
- the posterior uncertainty band
  $$
  \mu(x)\pm 2\sigma(x)
  $$
  is shown,
- and the observed data are overlaid.

So the only thing changing across the four panels is the kernel hyperparameters.

This makes it possible to see how those modelling assumptions affect the posterior even when the data are held fixed.

---

### The effect of the lengthscale

The **lengthscale** controls how quickly correlation decays with distance in the RBF kernel:

$$
k(x, x') = \sigma_f^2 \exp\left(-\frac{(x-x')^2}{2\ell^2}\right).
$$

This has a direct effect on the posterior.

- A **small lengthscale** means that influence is more local.
  Observations affect the function strongly only in nearby regions.
  As a result, the posterior mean can vary more rapidly, and the uncertainty can form more localised structures.
- A **large lengthscale** means that influence spreads more broadly.
  Observations affect wider regions of the input space.
  As a result, the posterior mean becomes smoother and changes more gradually across the domain.

So in practical terms:

> **lengthscale controls how locally or globally the data influence the posterior.**

This is why the panels with smaller $\ell$ tend to show more rapid variation, while the panels with larger $\ell$ produce smoother posterior curves.

---

### The effect of the variance

The **variance** parameter controls the vertical scale of function variation:

$$
\sigma_f^2.
$$

This affects the posterior in two related ways.

- A **smaller variance** gives a more conservative covariance structure.
  The resulting posterior uncertainty band is typically smaller in amplitude.
- A **larger variance** allows larger function excursions and larger uncertainty away from strongly constrained regions.

So in practical terms:

> **variance controls the scale of the posterior’s variation and the overall size of its uncertainty band.**

This is why increasing the variance tends to make the uncertainty region visually larger and allows the posterior to move more strongly in weakly constrained parts of the domain.

---

### How to read the four-panel figure

This layout is useful because it separates the effects of the two hyperparameters.

- Comparing **top vs bottom** isolates the effect of **lengthscale**
- Comparing **left vs right** isolates the effect of **variance**

So the figure shows that the kernel hyperparameters do not only affect the prior.
They also continue to shape the posterior after data has been observed.

That is an important lesson.

The posterior is not determined by the observations alone.
It is determined by the combination of:
- the observations
- and the assumptions built into the kernel

---

### Why this matters conceptually

This figure makes a subtle but important point:

> **Gaussian Process regression is not assumption-free.**

Even after conditioning on data, the posterior still reflects the kernel’s notion of smoothness, scale, and correlation.

So the hyperparameters are not minor technical details.
They directly influence:

- how the posterior interpolates between observations,
- how flexible or smooth the mean prediction is,
- and how uncertainty behaves in less constrained regions.

This is why kernel choice and hyperparameter choice matter so much in practice.

---

### Why this matters for Bayesian Optimisation

In Bayesian Optimisation, the surrogate model is used not only to fit the existing data, but also to decide where to evaluate next.

That means the shape of the posterior mean and the size of the posterior uncertainty band can directly affect:

- which regions appear promising,
- which regions appear uncertain,
- and how acquisition functions behave.

So hyperparameters influence not just the regression picture, but also the optimisation behaviour that follows from it.

---

### Key takeaway

> The kernel hyperparameters shape the Gaussian Process posterior even after conditioning on data.

In particular:

- the **lengthscale** controls how smooth and nonlocal the posterior behaviour is
- the **variance** controls the scale of posterior variation and uncertainty

This means that a GP posterior is not just “the data speaking for themselves.”
It is the result of data being interpreted through the assumptions encoded by the kernel.

In [ ]:
settings = [
    (0.3, 0.5),
    (0.3, 1.5),
    (1.2, 0.5),
    (1.2, 1.5),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True, sharey=True)
axes = axes.flatten()

for ax, (ell, var) in zip(axes, settings):
    mu_ell, var_ell, _ = gp_posterior(x_train, y_train, x_dense, lengthscale=ell, variance=var, noise=1e-4)
    sigma_ell = torch.sqrt(var_ell)

    ax.plot(x_dense.numpy(), y_dense.numpy(), linewidth=2.0, label="True objective")
    ax.plot(x_dense.numpy(), mu_ell.numpy(), linewidth=2.0, linestyle="--", label="GP mean")
    ax.fill_between(
        x_dense.numpy(),
        (mu_ell - 2.0 * sigma_ell).numpy(),
        (mu_ell + 2.0 * sigma_ell).numpy(),
        alpha=0.2,
        label=r"$\mu(x)\pm 2\sigma(x)$"
    )
    ax.scatter(x_train.numpy(), y_train.numpy(), s=40, zorder=3, label="Observations")
    ax.set_title(f"lengthscale={ell}, variance={var}", fontsize=18, fontweight="bold")
    ax.set_xlabel("x", fontsize=22, fontweight="bold")
    ax.set_ylabel("f(x)", fontsize=22, fontweight="bold")
    style_ax(ax)

axes[0].legend(prop={"size": 10, "weight": "bold"})
plt.tight_layout()
plt.show()

## 13. Adding one observation outside the originally observed region

The previous figure showed that increasing the total number of observations improves the Gaussian Process posterior.

This cell refines that idea in a more sequential and strategically meaningful way.

Instead of comparing two different datasets all at once, we now ask:

> **What happens if we add one new observation outside the region that was originally observed?**

This is an important question for Bayesian Optimisation, because in practice we do not usually collect a large batch of extra data at once.
We add observations one at a time, and where we place those observations matters.

---

### What the code does

We begin with a base training set of 8 observations in the interval

$$
[-2,\;2].
$$

Using those observations, we compute the GP posterior mean and uncertainty:

$$
\mu_{\mathrm{before}}(x), \qquad \sigma_{\mathrm{before}}(x).
$$

We then add one new observation at

$$
x_{\mathrm{new}} = 2.7,
$$

which lies **outside** the original observed region.

The true objective is evaluated there to obtain

$$
y_{\mathrm{new}} = f(x_{\mathrm{new}}).
$$

That new point is appended to the training data, and the GP posterior is recomputed, giving

$$
\mu_{\mathrm{after}}(x), \qquad \sigma_{\mathrm{after}}(x).
$$

The two panels compare the posterior:

- **before** adding the outside-region point
- **after** adding it

---

### Why this experiment is important

Up to this point, we have mainly seen that:
- the GP posterior is more confident near observed data
- and more uncertain away from them

This cell shows the natural next consequence:

> if we add data in a region that was previously weakly constrained, the posterior should update most strongly there.

That is exactly what happens in Gaussian Process regression.

A new observation outside the original training interval provides fresh information in a region where the model previously had relatively little support.

So this is not just “one more point.”
It is a demonstration that **where** a new observation is placed affects how the surrogate improves.

---

### How to interpret the two panels

#### Before adding the new point
The left panel shows the GP posterior built only from observations in

$$
[-2,\;2].
$$

Outside that interval, especially toward the right-hand side, the posterior is more weakly constrained.
The GP can still produce a mean prediction there, but its confidence is lower and its behaviour is more strongly governed by the prior assumptions.

#### After adding the new point
The right panel shows the posterior after including the observation at

$$
x=2.7.
$$

Now the GP has direct information in that previously underconstrained region.

As a result:

- the posterior mean adjusts in the neighbourhood of the new point
- the uncertainty band contracts locally around that region
- and the model becomes more informed beyond the original observation interval

So the figure shows that the GP posterior does not improve uniformly everywhere.
It improves most strongly where the new information actually arrives.

---

### The mathematical idea behind the update

This behaviour comes directly from the posterior equations.

When the new point is added, the training covariance matrix

$$
K_{xx}
$$

is enlarged to include the new observation, and the posterior quantities

$$
\mu(x)
\quad\text{and}\quad
\Sigma(x)
$$

are recomputed.

Because the kernel correlates nearby points, the new observation influences not only the exact location

$$
x_{\mathrm{new}},
$$

but also a neighbourhood around it.

So mathematically, the update works by:

- adding a new data constraint to the posterior mean
- and reducing the posterior covariance in the region that is now better supported

This is why the posterior mean shifts and the uncertainty narrows near the new point.

---

### Why this matters for Bayesian Optimisation

This is one of the most important transitions in the notebook.

Up to this point, the Gaussian Process has been presented mainly as a static surrogate model.

Here, it begins to behave like a **sequentially updated surrogate**, which is exactly how it is used in Bayesian Optimisation.

The key lesson is:

> **a new observation is valuable not only because it reveals one more function value, but because it reshapes the posterior in a strategically useful region.**

That is the essence of data-efficient optimisation.

When evaluations are expensive, it matters enormously whether the next point is chosen in a place that meaningfully improves the model.

---

### Connection to the previous section

The previous section showed:

- more observations improve the GP posterior

This section adds a deeper and more BO-relevant refinement:

- a carefully placed new observation can improve the posterior specifically where the model was previously weak or uncertain

So this figure is the sequential version of the same idea.

---

### Key takeaway

> Adding one observation outside the originally observed region can significantly improve the Gaussian Process posterior in that region.

This happens in two coupled ways:

- the posterior mean is adjusted using the new information
- the posterior uncertainty is reduced locally near the new point

This is exactly why Gaussian Processes are such powerful surrogate models for Bayesian Optimisation: they update both prediction and confidence in a way that makes the value of each new evaluation explicit.

In [ ]:
x_train_base = torch.linspace(-2, 2, 8)
y_train_base = expensive_objective(x_train_base)

mu_before, var_before, _ = gp_posterior(
    x_train_base, y_train_base, x_dense,
    lengthscale=0.8, variance=1.0, noise=1e-4
)
sigma_before = torch.sqrt(var_before)

# Add one new observation outside the original observed region
x_new = torch.tensor([2.7])
y_new = expensive_objective(x_new)

x_train_updated = torch.cat([x_train_base, x_new])
y_train_updated = torch.cat([y_train_base, y_new])

mu_after, var_after, _ = gp_posterior(
    x_train_updated, y_train_updated, x_dense,
    lengthscale=0.8, variance=1.0, noise=1e-4
)
sigma_after = torch.sqrt(var_after)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

axes[0].plot(x_dense.numpy(), y_dense.numpy(), linewidth=2.0, label="True objective")
axes[0].plot(x_dense.numpy(), mu_before.numpy(), linewidth=2.0, linestyle="--", label="GP mean")
axes[0].fill_between(
    x_dense.numpy(),
    (mu_before - 2.0 * sigma_before).numpy(),
    (mu_before + 2.0 * sigma_before).numpy(),
    alpha=0.2,
    label=r"$\mu(x)\pm 2\sigma(x)$"
)

axes[0].scatter(x_train_base.numpy(), y_train_base.numpy(), s=40, zorder=3, label="Original observations")
axes[0].set_title("Before adding outside-region observation", fontsize=18, fontweight="bold")
axes[0].set_xlabel("x", fontsize=22, fontweight="bold")
axes[0].set_ylabel("f(x)", fontsize=22, fontweight="bold")
axes[0].legend(prop={"size": 10, "weight": "bold"})
style_ax(axes[0])

axes[1].plot(x_dense.numpy(), y_dense.numpy(), linewidth=2.0, label="True objective")
axes[1].plot(x_dense.numpy(), mu_after.numpy(), linewidth=2.0, linestyle="--", label="GP mean")
axes[1].fill_between(
    x_dense.numpy(),
    (mu_after - 2.0 * sigma_after).numpy(),
    (mu_after + 2.0 * sigma_after).numpy(),
    alpha=0.2,
    label=r"$\mu(x)\pm 2\sigma(x)$"
)

axes[1].scatter(x_train_updated.numpy(), y_train_updated.numpy(), s=40, zorder=3, label="Original observations")
axes[1].scatter(x_new.numpy(), y_new.numpy(), s=90, marker="*", zorder=4, label="New outside-region point")
axes[1].set_title("After adding outside-region observation", fontsize=18, fontweight="bold")
axes[1].set_xlabel("x", fontsize=22, fontweight="bold")
axes[1].set_ylabel("f(x)", fontsize=22, fontweight="bold")
axes[1].legend(prop={"size": 10, "weight": "bold"})
style_ax(axes[1])

plt.tight_layout()
plt.show()

In [ ]:
mu_gp, var_gp, _ = gp_posterior(x_train, y_train, x_dense, lengthscale=0.8, variance=1.0, noise=1e-4)
sigma_gp = torch.sqrt(var_gp)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(x_dense.numpy(), y_dense.numpy(), linewidth=2.0, label="true objective")
ax.plot(x_dense.numpy(), mu_gp.numpy(), linewidth=2.0, linestyle="--", label="GP mean")
ax.fill_between(
    x_dense.numpy(),
    (mu_gp - 2.0 * sigma_gp).numpy(),
    (mu_gp + 2.0 * sigma_gp).numpy(),
    alpha=0.2,
    label=r"$\mu(x)\pm 2\sigma(x)$"
)
ax.scatter(x_train.numpy(), y_train.numpy(), s=40, zorder=3, label="observations")

ax.set_title("A Gaussian Process gives both prediction and uncertainty", fontsize=18, fontweight="bold")
ax.set_xlabel("x", fontsize=22, fontweight="bold")
ax.set_ylabel("f(x)", fontsize=22, fontweight="bold")
ax.legend(prop={"size": 10, "weight": "bold"})
style_ax(ax)
plt.show()

## 14. Sequentially expanding the observed region

The previous cell showed that adding one observation outside the originally observed region can improve the Gaussian Process posterior locally.

This cell extends that idea further by showing a genuinely **sequential update process**.

Instead of adding just one new point, we now add several new observations one at a time outside the initial data-supported region and observe how the posterior evolves after each update.

This is a much closer reflection of how Gaussian Processes are used in model-guided optimisation.

---

### What the code does

We begin with an initial training set of 8 observations in the interval

$$
[-1.5,\;1.5].
$$

Using those observations, we compute the initial GP posterior mean and uncertainty:

$$
\mu(x), \qquad \sigma(x).
$$

We then sequentially add the following new observations:

$$
x = -3.0,\; 3.0,\; -2.5,\; -2.0,\; 2.0.
$$

After each new point is added:

- the true objective is evaluated at that point,
- the training set is updated,
- the GP posterior is recomputed,
- and the updated mean and uncertainty band are plotted.

So the six panels show a step-by-step record of how the surrogate changes as the observed region expands outward.

---

### How to interpret the figure

The first panel shows the starting situation:

- observations are confined to the central interval
  $$
  [-1.5,\;1.5],
  $$
- the posterior mean is most reliable there,
- and uncertainty grows outside that region.

The next panels show what happens as new outside-region points are added.

Each new point does two things:

- it pulls the posterior mean toward the newly observed function value,
- and it reduces uncertainty in the surrounding region.

Because the points are added sequentially, the figure makes visible a very important feature of Gaussian Processes:

> **posterior improvement is incremental and location-dependent.**

The model does not improve everywhere equally.
It improves most strongly near the locations where new information is acquired.

---

### Why the order of added points matters

The new points are not added all at once.
They are added in a specific sequence:

1. first at the far left boundary
2. then at the far right boundary
3. then at intermediate outer points on the left
4. then at a closer point on the left
5. and finally at a corresponding point on the right

This creates a useful visual story.

The earliest added points reduce the most extreme extrapolation uncertainty at the outer edges.
The later added points then refine the posterior between the original central region and those newly observed boundary locations.

So the figure shows not only that more data helps, but also that:

> **the spatial placement of new data determines how the surrogate improves over time.**

---

### Why this matters conceptually

This cell is important because it shifts the perspective from:

- a GP as a static regression model

to

- a GP as a **sequentially updated surrogate model**.

That is exactly the viewpoint needed for Bayesian Optimisation.

In expensive optimisation, we do not usually get all the data at once.
We choose one point, evaluate it, update the surrogate, and then decide what to do next.

So this figure is not just an illustration of GP regression.
It is a visual demonstration of the iterative learning loop:

1. start with limited data
2. identify regions of weak support
3. add new observations
4. update the surrogate
5. reduce uncertainty and improve prediction over time

---

### Why this matters for optimisation

This sequential update behaviour is one of the main reasons Gaussian Processes are so useful.

Each new evaluation does not just reveal a single new function value.
It changes the model’s understanding of the objective and alters the uncertainty structure that will shape future decisions.

So this cell makes an important optimisation lesson visible:

> **the value of a new observation lies not only in the value observed at that point, but in how that point changes the posterior mean and uncertainty in neighbouring regions.**

That is the core mechanism behind data-efficient model-guided search.

---

### Key takeaway

> Sequentially adding observations outside the original observed region causes the Gaussian Process posterior to expand, adapt, and become more confident in those newly supported regions.

This figure shows the GP behaving as it should:

- the mean prediction updates as new information arrives,
- the uncertainty shrinks where support improves,
- and the surrogate gradually becomes more informative over a larger portion of the domain.

This is one of the clearest demonstrations in the notebook that Gaussian Processes are not just flexible regression tools, but dynamic surrogate models that improve step by step as new data are gathered.

In [ ]:
x_seq = torch.linspace(-1.5, 1.5, 8)
y_seq = expensive_objective(x_seq)

new_points = [
    torch.tensor([-3.0]),
    torch.tensor([3.0]),
    torch.tensor([-2.5]),
    torch.tensor([-2.0]),
    torch.tensor([2.0]),
]

panel_titles = [
    "Start: observations in [-1.5, 1.5]",
    "After adding x = -3",
    "After adding x = 3",
    "After adding x = -2.5",
    "After adding x = -2",
    "After adding x = 2",
]

fig, axes = plt.subplots(2, 3, figsize=(16, 10), sharex=True, sharey=True)
axes = axes.flatten()

# Panel 1: initial posterior
mu_seq, var_seq, _ = gp_posterior(
    x_seq, y_seq, x_dense,
    lengthscale=0.8, variance=1.0, noise=1e-4
)
sigma_seq = torch.sqrt(var_seq)

axes[0].plot(x_dense.numpy(), y_dense.numpy(), linewidth=2.0, label="True objective")
axes[0].plot(x_dense.numpy(), mu_seq.numpy(), linewidth=2.0, linestyle="--", label="GP mean")
axes[0].fill_between(
    x_dense.numpy(),
    (mu_seq - 2.0 * sigma_seq).numpy(),
    (mu_seq + 2.0 * sigma_seq).numpy(),
    alpha=0.2,
    label=r"$\mu(x)\pm 2\sigma(x)$"
)
axes[0].scatter(x_seq.numpy(), y_seq.numpy(), s=40, zorder=3, label="Observations")
axes[0].set_title(panel_titles[0], fontsize=18, fontweight="bold")
axes[0].set_xlabel("x", fontsize=22, fontweight="bold")
axes[0].set_ylabel("f(x)", fontsize=22, fontweight="bold")
style_ax(axes[0])

# Panels 2–6: sequentially add outside-region points
for ax, x_new, title in zip(axes[1:6], new_points, panel_titles[1:]):
    y_new = expensive_objective(x_new)

    x_seq = torch.cat([x_seq, x_new])
    y_seq = torch.cat([y_seq, y_new])

    mu_seq, var_seq, _ = gp_posterior(
        x_seq, y_seq, x_dense,
        lengthscale=0.8, variance=1.0, noise=1e-4
    )
    sigma_seq = torch.sqrt(var_seq)

    ax.plot(x_dense.numpy(), y_dense.numpy(), linewidth=2.0, label="True objective")
    ax.plot(x_dense.numpy(), mu_seq.numpy(), linewidth=2.0, linestyle="--", label="GP mean")
    ax.fill_between(
        x_dense.numpy(),
        (mu_seq - 2.0 * sigma_seq).numpy(),
        (mu_seq + 2.0 * sigma_seq).numpy(),
        alpha=0.2,
        label=r"$\mu(x)\pm 2\sigma(x)$"
    )
    ax.scatter(x_seq.numpy(), y_seq.numpy(), s=40, zorder=3, label="Observations")
    ax.scatter(x_new.numpy(), y_new.numpy(), s=90, marker="*", zorder=4, label="Newest point")
    ax.set_title(title, fontsize=18, fontweight="bold")
    ax.set_xlabel("x", fontsize=22, fontweight="bold")
    ax.set_ylabel("f(x)", fontsize=22, fontweight="bold")
    style_ax(ax)

axes[0].legend(prop={"size": 10, "weight": "bold"})
plt.tight_layout()
plt.show()

## 🧭 Closing Remarks

In this tutorial, we moved from the intuitive idea of uncertainty developed in **Tutorial 2** to a more principled probabilistic surrogate model:

> **the Gaussian Process.**

That is the central conceptual achievement of the notebook.

Earlier, we saw that a useful surrogate should provide not only a prediction of the unknown objective, but also a measure of how uncertain that prediction is.
Here, we learned where those two quantities come from in a mathematically coherent model.

A Gaussian Process gives us:

- a predictive mean
  $$
  \mu(x),
  $$
- and a predictive uncertainty
  $$
  \sigma(x),
  $$

by treating the unknown objective not as a single fixed curve, but as a **distribution over functions**.

---

The notebook developed this idea in stages.

We began by introducing the kernel as the object that encodes similarity between inputs and therefore shapes the covariance structure of the surrogate.

That led naturally to the **GP prior**, which represents the model’s belief about what kinds of functions are plausible before seeing any data.

We then saw how conditioning on observed points transforms that prior into a **GP posterior**, whose mean is informed by the data and whose uncertainty narrows where the function has been better constrained.

This is the key transition:

> the Gaussian Process does not abandon uncertainty after seeing data — it updates uncertainty into a more informed posterior form.

That is what makes it such a natural surrogate model.

---

A consistent pattern emerged throughout the notebook:

- the posterior mean is pulled toward the observations,
- the uncertainty is reduced where the model has support,
- and uncertainty remains larger in weakly observed or extrapolative regions.

This made it clear that the GP posterior is not just a smooth fit through data.

It is a structured probabilistic object that tells us both:
- what the model currently believes,
- and where that belief is still fragile.

That is why the pair

$$
\mu(x), \qquad \sigma(x)
$$

is so important.

It captures both **prediction** and **confidence** in a single surrogate model.

---

We also saw that the Gaussian Process is shaped not only by the observed data, but also by the assumptions encoded in the kernel.

In particular, the RBF hyperparameters affected:

- the smoothness of the posterior,
- the scale of its variation,
- and the size and structure of the uncertainty band.

So the GP is not an assumption-free model.
It is a model in which prior beliefs and observed data interact.

That is an important lesson, because it shows that surrogate modelling is never just about fitting what has been seen.
It is also about deciding what kinds of unseen behaviour the model is prepared to consider plausible.

---

The final sections made an especially important point for optimisation.

By adding observations sequentially, we saw that every new evaluation changes the surrogate in two coupled ways:

- it updates the mean prediction,
- and it reduces uncertainty in the surrounding region.

This makes clear that Gaussian Processes are not static regression tools.
They are **sequentially updated models of knowledge**.

That is exactly the perspective needed for Bayesian Optimisation.

A new evaluation is valuable not only because it reveals one more function value, but because it changes the posterior and therefore changes what the model knows, what it does not know, and where it should look next.

---

If there is one central takeaway from this tutorial, it is this:

> **A Gaussian Process is a probabilistic surrogate model that turns sparse observations into both a prediction and a principled measure of uncertainty.**

That is why Gaussian Processes sit at the heart of Bayesian Optimisation.

They provide exactly the ingredients needed to make intelligent sequential decisions under limited evaluation budgets:
- a model of the objective,
- a model of uncertainty,
- and a way to update both as new data arrives.

---

This prepares us directly for the next part of the story.

In the next tutorial, we will stop treating
$$
\mu(x)
$$
and
$$
\sigma(x)
$$
as ends in themselves, and begin using them to answer the real optimisation question:

> **Given a Gaussian Process surrogate, where should we evaluate next?**

That is the point where surrogate modelling becomes **Bayesian Optimisation proper**.